# 직접 수집 5 · 보고서 — AI 요약

수집·분석 결과(통계)를 **LLM이 해석**해 보고서를 자동 작성합니다.
- 로컬 **Ollama**(OpenAI 호환) **qwen3:8b** 사용, 엔드포인트 `http://localhost:11434/v1`
- ⚠️ Ollama 서버가 켜져 있어야 합니다. (`ollama serve`, 모델 준비: `ollama pull qwen3:8b`)

In [1]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # wiki_crawling.py 있는 프로젝트 루트 탐색
    if os.path.isfile("wiki_crawling.py"):
        break
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 2                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

작업 루트: /Users/sumin/Desktop/유망 아이템/wiki_web
대상 문서: Quantum computing | 작업 폴더: runs/Quantum_computing


## 통계 결과 로드 (04 노트북 산출물)

In [2]:
if not os.path.isfile(STATS):
    raise FileNotFoundError("통계 파일이 없습니다. 04 노트북을 먼저 실행하세요.")
stats = pd.read_excel(STATS)
# 기술집약도 기준 상위 아이템을 보고서 근거로
cols = [c for c in ["title", "기술집약도", "수요부상성", "공급부상성"] if c in stats.columns]
top = stats[cols].sort_values("기술집약도", ascending=False).head(12)
rows = top.to_dict("records")
print("보고서 근거 상위", len(rows), "건")
top

보고서 근거 상위 12 건


,title,기술집약도,수요부상성,공급부상성
6,D-Wave Systems,0.021845,0.122407,0.050701
22,Magic (quantum information),0.021323,1.000000,1.000000
9,Fidelity of quantum states,0.016305,0.131381,0.066225
11,Formal science,0.016305,0.077906,0.010870
51,Quantum logic,0.016305,0.078652,0.017271
49,Quantum compass,0.016305,0.222317,0.086207
62,Sun–Ni law,0.016305,0.228696,0.027027
52,Quantum metrology,0.016305,0.088405,0.105263
44,QBism,0.016305,0.000000,0.064295
32,Noisy intermediate-scale quantum computing,0.016305,1.000000,0.306452


## LLM 호출 → 보고서

In [ ]:
# 로컬 LLM 직접 호출 (Ollama, OpenAI 호환 API) — 'openai' 패키지 불필요, requests만 사용
import json, requests

LLM_URL = "http://localhost:11434/v1/chat/completions"   # Ollama 엔드포인트
MODEL   = "qwen3:8b"                                      # `ollama list` 에 보이는 모델명

system = ("당신은 KISTI의 기술기획 분석가입니다. 위키 네트워크에서 수집한 지표 데이터를 근거로 "
          "교육생이 이해하기 쉬운 한국어 유망아이템 보고서를 작성합니다. 데이터에 없는 수치는 지어내지 말고, "
          "기술집약도·수요/공급 부상성의 의미를 풀어 해석하세요. 마크다운으로 간결하게.")
user = (f"시드 문서: {SEED}\n\n[연관 문서 지표 상위 (JSON)]\n"
        f"{json.dumps(rows, ensure_ascii=False, indent=2)}\n\n"
        "다음 보고서를 작성하세요:\n## 1. 분석 개요\n## 2. 주목 아이템 TOP 3 해석\n## 3. 종합 시사점")

payload = {
    "model": MODEL,
    "messages": [{"role": "system", "content": system},
                 {"role": "user", "content": user}],
    "temperature": 0.4,
    "max_tokens": 1800,
    "chat_template_kwargs": {"enable_thinking": False},   # Qwen3 thinking 끄기
}
resp = requests.post(LLM_URL, json=payload, timeout=120)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

**정리** — 위키 원본 크롤링부터 수집→정제→결과→보고서까지 전 과정을 코드로 시연했습니다. 🎉